# Stellar-stream PyG label sanity checks

Explore graphs in:

`.../streams_per_region_iter3_pm_stepping2_k4_edges_pmcoh/pyg_data_pmcoh_dir0p97_dlog0p25`

*(Note: the build tag is `dlog0p25`, not `dlog0p025`.)*

Expected label semantics (from `stream_graph_dataset.py`):
- **`y[i] = 1`** iff star `i` has `stream_label ==` the graph's target stream
- **`edge_y[e] = 1`** iff **both** endpoints of edge `e` are target members
- **`train_mask`**: obvious stream (+ optional obvious background)
- **`val_mask` / `test_mask`**: holdout of non-obvious target members (or a fraction of all targets if everything is "obvious")

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

%matplotlib inline

DATA_DIR = Path(
    "/projects/BHANIN/adri_gage_gnn_streams/gnn_streams/mocks/"
    "streams_per_region_iter3_pm_stepping2_k4_edges_pmcoh/"
    "pyg_data_pmcoh_dir0p97_dlog0p25"
)
PARENT = DATA_DIR.parent  # parquets + raw edge .pt live one level up

assert DATA_DIR.is_dir(), f"missing {DATA_DIR}"
FILES = sorted(DATA_DIR.glob("stream_*.pt"))
print(f"{len(FILES)} graphs in {DATA_DIR}")
print("examples:", [p.name for p in FILES[:8]])

39 graphs in /projects/BHANIN/adri_gage_gnn_streams/gnn_streams/mocks/streams_per_region_iter3_pm_stepping2_k4_edges_pmcoh/pyg_data_pmcoh_dir0p97_dlog0p25
examples: ['stream_2df__ngc1261_p_0.pt', 'stream_2df__ngc1851_p_0.pt', 'stream_2df__ngc1904_p_0.pt', 'stream_AAU.pt', 'stream_Chenab.pt', 'stream_Elqui.pt', 'stream_Epod.pt', 'stream_Fornax.pt']


## Schema peek on one graph

In [2]:
def load_graph(path: Path):
    return torch.load(path, map_location="cpu", weights_only=False)


g0 = load_graph(FILES[0])
print(FILES[0].name)
print("keys:", sorted(g0.keys()))
for k in sorted(g0.keys()):
    v = g0[k]
    if torch.is_tensor(v):
        print(f"  {k:18s} shape={tuple(v.shape)} dtype={v.dtype}")
    else:
        print(f"  {k:18s} {type(v).__name__}={v}")
print("feature_names:", g0.feature_names)
print("edge_attr_names:", g0.edge_attr_names)

/home/jdezoort/.conda/envs/mamba-env/lib/python3.10/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /home/jdezoort/.conda/envs/mamba-env/lib/python3.10/site-packages/libpyg.so: undefined symbol: _ZN3c1010Dispatcher17runRecordFunctionERN2at14RecordFunctionESt17reference_wrapperIKNS_14FunctionSchemaEENS_11DispatchKeyE
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/home/jdezoort/.conda/envs/mamba-env/lib/python3.10/site-packages/torch_geometric/typing.py:72: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/jdezoort/.conda/envs/mamba-env/lib/python3.10/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/jdezoort/.conda/envs/mamba-env/lib/python3.10/site-packages/torch_geometric/typing.py:110: UserWarning: An issue 

stream_2df__ngc1261_p_0.pt
keys: ['edge_attr', 'edge_attr_names', 'edge_index', 'edge_index_path', 'edge_variant', 'edge_y', 'feature_names', 'num_nodes', 'parquet_path', 'safe_name', 'stream_label', 'test_mask', 'train_mask', 'val_mask', 'x', 'y']
  edge_attr          shape=(7386, 4) dtype=torch.float32
  edge_attr_names    list=['round', 'knn_dist', 'dir_cos', 'dlog_speed']
  edge_index         shape=(2, 7386) dtype=torch.int64
  edge_index_path    str=streams_per_region_iter3_pm_stepping2_k4_edges_pmcoh/stream_2df__ngc1261_p_0_E7386_pmcoh_dir0p97_dlog0p25_edge_index.pt
  edge_variant       str=pmcoh_dir0p97_dlog0p25
  edge_y             shape=(7386,) dtype=torch.int64
  feature_names      list=['phi', 'lam', 'pm_phi', 'pm_lam', 'parallax_gaia', 'psf_mag_aper_8_g_corrected_des', 'psf_mag_aper_8_r_corrected_des']
  num_nodes          int=77024
  parquet_path       str=streams_per_region_iter3_pm_stepping2_k4_edges_pmcoh/stream_2df__ngc1261_p_0.parquet
  safe_name          str=2df__ngc

## Catalog: node / edge positivity + mask bookkeeping

Flags of interest:
- `edge_y` all zeros while `y` has positives → no stream–stream edges survived PM-coherence pruning
- `edge_y` disagrees with recomputed `(y[src] & y[dst])` → label bug
- overlapping train/val/test masks → split bug
- positives missing from all masks → unlabeled members (OK for semi-supervised pool, but worth counting)

In [3]:
import gc

rows = []
for path in FILES:
    g = load_graph(path)
    y = g.y.long()
    ey = g.edge_y.long()
    src, dst = g.edge_index
    ey_re = ((y[src] == 1) & (y[dst] == 1)).long()

    tr, va, te = g.train_mask, g.val_mask, g.test_mask
    mask_sum = tr.long() + va.long() + te.long()

    n = int(g.num_nodes)
    e = int(g.edge_index.size(1))
    y_pos = int((y == 1).sum())
    ey_pos = int((ey == 1).sum())

    # endpoint classes among existing edges
    ss = int(((y[src] == 1) & (y[dst] == 1)).sum())
    so = int((((y[src] == 1) & (y[dst] == 0)) | ((y[src] == 0) & (y[dst] == 1))).sum())
    bb = int(((y[src] == 0) & (y[dst] == 0)).sum())

    rows.append(
        dict(
            file=path.name,
            stream=str(g.stream_label),
            safe=str(g.safe_name),
            variant=str(getattr(g, "edge_variant", "")),
            n_nodes=n,
            n_edges=e,
            y_pos=y_pos,
            y_frac=y_pos / n if n else np.nan,
            edge_y_pos=ey_pos,
            edge_y_recomputed=ss,
            edge_y_mismatch=int((ey != ey_re).sum()),
            edges_SS=ss,
            edges_SO=so,
            edges_BB=bb,
            train=int(tr.sum()),
            val=int(va.sum()),
            test=int(te.sum()),
            mask_overlap=int((mask_sum > 1).sum()),
            mask_cover=int((mask_sum > 0).sum()),
            y_pos_in_train=int((y == 1)[tr].sum()),
            y_pos_in_val=int((y == 1)[va].sum()),
            y_pos_in_test=int((y == 1)[te].sum()),
            y_pos_unmasked=int(((y == 1) & ~(tr | va | te)).sum()),
            y_neg_in_valtest=int(((y == 0) & (va | te)).sum()),
            train_pos_frac=float(((y == 1)[tr]).float().mean()) if tr.any() else np.nan,
            y_unique=tuple(sorted(torch.unique(y).tolist())),
            ey_unique=tuple(sorted(torch.unique(ey).tolist())),
            parquet_exists=Path(g.parquet_path).is_file(),
        )
    )
    print(
        f"{path.name:40s} y+={y_pos:6d}/{n:7d} ({100*y_pos/n:5.2f}%)  "
        f"edge_y+={ey_pos:5d}/{e:5d} (SS/SO/BB={ss}/{so}/{bb})  "
        f"mism={rows[-1]['edge_y_mismatch']}  "
        f"masks train/val/test={rows[-1]['train']}/{rows[-1]['val']}/{rows[-1]['test']}"
    )
    del g, y, ey, src, dst, ey_re, tr, va, te, mask_sum
    gc.collect()

catalog = pd.DataFrame(rows)
catalog

stream_2df__ngc1261_p_0.pt               y+=   216/  77024 ( 0.28%)  edge_y+=  433/ 7386 (SS/SO/BB=433/3060/3893)  mism=0  masks train/val/test=26650/22/21
stream_2df__ngc1851_p_0.pt               y+=   247/ 118925 ( 0.21%)  edge_y+=  615/ 9397 (SS/SO/BB=615/3830/4952)  mism=0  masks train/val/test=17496/24/25
stream_2df__ngc1904_p_0.pt               y+=   314/ 202655 ( 0.15%)  edge_y+=  759/12038 (SS/SO/BB=759/4898/6381)  mism=0  masks train/val/test=13622/32/31
stream_AAU.pt                            y+=  3461/ 343498 ( 1.01%)  edge_y+= 4708/115603 (SS/SO/BB=4708/47679/63216)  mism=0  masks train/val/test=166557/346/346
stream_Chenab.pt                         y+=  2497/ 591883 ( 0.42%)  edge_y+= 3271/88000 (SS/SO/BB=3271/36905/47824)  mism=0  masks train/val/test=175680/250/249
stream_Elqui.pt                          y+=  1015/ 249036 ( 0.41%)  edge_y+= 1346/33254 (SS/SO/BB=1346/13810/18098)  mism=0  masks train/val/test=119191/102/101
stream_Epod.pt                           y+= 

,file,stream,safe,variant,n_nodes,n_edges,y_pos,y_frac,edge_y_pos,edge_y_recomputed,...,mask_cover,y_pos_in_train,y_pos_in_val,y_pos_in_test,y_pos_unmasked,y_neg_in_valtest,train_pos_frac,y_unique,ey_unique,parquet_exists
0,stream_2df__ngc1261_p_0.pt,2df ngc1261 p 0,2df__ngc1261_p_0,pmcoh_dir0p97_dlog0p25,77024,7386,216,0.002804,433,433,...,26693,173,22,21,0,0,0.006492,"(0, 1)","(0, 1)",False
1,stream_2df__ngc1851_p_0.pt,2df ngc1851 p 0,2df__ngc1851_p_0,pmcoh_dir0p97_dlog0p25,118925,9397,247,0.002077,615,615,...,17545,198,24,25,0,0,0.011317,"(0, 1)","(0, 1)",False
2,stream_2df__ngc1904_p_0.pt,2df ngc1904 p 0,2df__ngc1904_p_0,pmcoh_dir0p97_dlog0p25,202655,12038,314,0.001549,759,759,...,13685,251,32,31,0,0,0.018426,"(0, 1)","(0, 1)",False
3,stream_AAU.pt,AAU,AAU,pmcoh_dir0p97_dlog0p25,343498,115603,3461,0.010076,4708,4708,...,167249,2769,346,346,0,0,0.016625,"(0, 1)","(0, 1)",False
4,stream_Chenab.pt,Chenab,Chenab,pmcoh_dir0p97_dlog0p25,591883,88000,2497,0.004219,3271,3271,...,176179,1998,250,249,0,0,0.011373,"(0, 1)","(0, 1)",False
5,stream_Elqui.pt,Elqui,Elqui,pmcoh_dir0p97_dlog0p25,249036,33254,1015,0.004076,1346,1346,...,119394,812,102,101,0,0,0.006813,"(0, 1)","(0, 1)",False
6,stream_Epod.pt,Epod,Epod,pmcoh_dir0p97_dlog0p25,240712,46176,1289,0.005355,1646,1646,...,84235,1031,129,129,0,0,0.012277,"(0, 1)","(0, 1)",False
7,stream_Fornax.pt,Fornax,Fornax,pmcoh_dir0p97_dlog0p25,99924,2312,96,0.000961,120,120,...,38625,77,10,9,0,0,0.001995,"(0, 1)","(0, 1)",False
8,stream_Indus.pt,Indus,Indus,pmcoh_dir0p97_dlog0p25,569542,62146,2047,0.003594,3498,3498,...,163398,1638,204,205,0,0,0.010050,"(0, 1)","(0, 1)",False
9,stream_Jhelum.pt,Jhelum,Jhelum,pmcoh_dir0p97_dlog0p25,977569,179640,6335,0.006480,11288,11288,...,348256,5068,634,633,0,0,0.014606,"(0, 1)","(0, 1)",False


In [4]:
print("=== FAIL checks (should be empty / zero) ===")
print("edge_y mismatches:", int(catalog["edge_y_mismatch"].sum()))
print("mask overlaps:", int(catalog["mask_overlap"].sum()))
print("negatives in val/test:", int(catalog["y_neg_in_valtest"].sum()))
print("graphs with y_pos==0:", catalog.loc[catalog["y_pos"] == 0, "file"].tolist())
print(
    "graphs with y_pos>0 but edge_y_pos==0:",
    catalog.loc[(catalog["y_pos"] > 0) & (catalog["edge_y_pos"] == 0), "file"].tolist(),
)
print(
    "graphs with val+test==0 despite y_pos>0:",
    catalog.loc[
        (catalog["y_pos"] > 0) & (catalog["val"] + catalog["test"] == 0), "file"
    ].tolist(),
)

display(
    catalog[
        [
            "file",
            "y_pos",
            "y_frac",
            "edge_y_pos",
            "edges_SS",
            "edges_SO",
            "edges_BB",
            "y_pos_in_train",
            "y_pos_in_val",
            "y_pos_in_test",
            "y_pos_unmasked",
            "train_pos_frac",
        ]
    ].sort_values("y_frac", ascending=False)
)

=== FAIL checks (should be empty / zero) ===
edge_y mismatches: 0
mask overlaps: 0
negatives in val/test: 0
graphs with y_pos==0: []
graphs with y_pos>0 but edge_y_pos==0: []
graphs with val+test==0 despite y_pos>0: []


,file,y_pos,y_frac,edge_y_pos,edges_SS,edges_SO,edges_BB,y_pos_in_train,y_pos_in_val,y_pos_in_test,y_pos_unmasked,train_pos_frac
3,stream_AAU.pt,3461,0.010076,4708,4708,47679,63216,2769,346,346,0,0.016625
32,stream_Smcnod.pt,737,0.009405,1519,1519,11228,9944,590,74,73,0,0.025333
15,stream_MOCK_2067.pt,2224,0.008224,15611,15611,10518,6465,1468,378,378,0,0.016296
30,stream_Phoenix.pt,1141,0.006652,1543,1543,16913,22764,913,114,114,0,0.012531
9,stream_Jhelum.pt,6335,0.006480,11288,11288,76103,92249,5068,634,633,0,0.014606
25,stream_N1261.pt,1391,0.005385,1978,1978,19956,27346,1113,139,139,0,0.012848
6,stream_Epod.pt,1289,0.005355,1646,1646,19092,25438,1031,129,129,0,0.012277
26,stream_Ngc1851.pt,1722,0.004983,2406,2406,27406,36363,1378,172,172,0,0.028187
33,stream_T3e.pt,1704,0.004838,2272,2272,21240,26965,1363,170,171,0,0.010549
23,stream_MOCK_4626.pt,578,0.004655,5638,5638,1814,1337,323,128,127,0,0.005730


## Cross-check `y` against the region parquet

Rebuild node labels from `stream_label == data.stream_label` and compare to `data.y`.

In [5]:
pq_checks = []
for path in FILES:
    g = load_graph(path)
    pq = Path(g.parquet_path)
    if not pq.is_file():
        pq_checks.append(dict(file=path.name, ok=False, reason="missing parquet"))
        continue

    need = ["stream_label", "is_obvious_stream"]
    cols = [c for c in need + ["is_obvious_background", "phi", "lam"] if True]
    # read only columns that exist
    import pyarrow.parquet as pq_mod

    schema_names = set(pq_mod.read_schema(pq).names)
    use_cols = [c for c in ["stream_label", "is_obvious_stream", "is_obvious_background", "phi", "lam"] if c in schema_names]
    df = pd.read_parquet(pq, columns=use_cols)

    if len(df) != int(g.num_nodes):
        pq_checks.append(
            dict(
                file=path.name,
                ok=False,
                reason=f"len mismatch parquet={len(df)} graph={g.num_nodes}",
            )
        )
        continue

    is_target = (df["stream_label"].astype(str) == str(g.stream_label)).to_numpy()
    y_np = g.y.numpy()
    agree = bool(np.array_equal(is_target.astype(np.int64), y_np))

    # train_mask composition vs obvious flags
    obv_s = df["is_obvious_stream"].fillna(False).to_numpy(dtype=bool) if "is_obvious_stream" in df else None
    obv_b = (
        df["is_obvious_background"].fillna(False).to_numpy(dtype=bool)
        if "is_obvious_background" in df
        else None
    )
    tr = g.train_mask.numpy()
    train_vs_obv = None
    if obv_s is not None and obv_b is not None:
        # default construction: train ⊇ obvious stream ∪ obvious bg, except held-out targets removed
        expected_core = obv_s | obv_b
        # held-out targets should not be in train
        holdout = (g.val_mask | g.test_mask).numpy()
        expected_train = expected_core & ~holdout
        # if all targets were obvious, holdout was carved from targets and removed from train
        train_vs_obv = {
            "train_extra": int((tr & ~expected_train).sum()),
            "train_missing_vs_expected": int((expected_train & ~tr).sum()),
            "obv_stream_in_train": int((obv_s & tr).sum()),
            "obv_stream_total": int(obv_s.sum()),
            "obv_bg_in_train": int((obv_b & tr).sum()),
            "obv_bg_total": int(obv_b.sum()),
            "holdout_in_train": int((holdout & tr).sum()),
        }

    pq_checks.append(
        dict(
            file=path.name,
            stream=str(g.stream_label),
            ok=agree,
            n_disagree=int((is_target.astype(np.int64) != y_np).sum()),
            n_stream_parquet=int(is_target.sum()),
            n_y=int(y_np.sum()),
            n_unique_labels=int(df["stream_label"].nunique()),
            label_counts=df["stream_label"].value_counts().head(5).to_dict(),
            **(train_vs_obv or {}),
        )
    )

pq_df = pd.DataFrame(pq_checks)
print("y vs parquet disagreements:", int((~pq_df["ok"]).sum()) if "ok" in pq_df else "?")
display(pq_df.drop(columns=["label_counts"], errors="ignore"))
bad = pq_df.loc[~pq_df["ok"]]
if len(bad):
    display(bad)

y vs parquet disagreements: 39


,file,ok,reason
0,stream_2df__ngc1261_p_0.pt,False,missing parquet
1,stream_2df__ngc1851_p_0.pt,False,missing parquet
2,stream_2df__ngc1904_p_0.pt,False,missing parquet
3,stream_AAU.pt,False,missing parquet
4,stream_Chenab.pt,False,missing parquet
5,stream_Elqui.pt,False,missing parquet
6,stream_Epod.pt,False,missing parquet
7,stream_Fornax.pt,False,missing parquet
8,stream_Indus.pt,False,missing parquet
9,stream_Jhelum.pt,False,missing parquet


,file,ok,reason
0,stream_2df__ngc1261_p_0.pt,False,missing parquet
1,stream_2df__ngc1851_p_0.pt,False,missing parquet
2,stream_2df__ngc1904_p_0.pt,False,missing parquet
3,stream_AAU.pt,False,missing parquet
4,stream_Chenab.pt,False,missing parquet
5,stream_Elqui.pt,False,missing parquet
6,stream_Epod.pt,False,missing parquet
7,stream_Fornax.pt,False,missing parquet
8,stream_Indus.pt,False,missing parquet
9,stream_Jhelum.pt,False,missing parquet


## Spatial view for a few graphs

Plot `(phi, lam)` colored by node label / masks, and overlay edges colored by `edge_y`.

In [6]:
def plot_graph_labels(path: Path, max_edges: int = 4000, seed: int = 0):
    g = load_graph(path)
    df = pd.read_parquet(g.parquet_path, columns=["phi", "lam", "stream_label", "is_obvious_stream"])
    phi, lam = df["phi"].to_numpy(), df["lam"].to_numpy()
    y = g.y.numpy()
    tr, va, te = g.train_mask.numpy(), g.val_mask.numpy(), g.test_mask.numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=True, sharey=True)

    # panel 0: background vs target
    ax = axes[0]
    ax.scatter(phi[y == 0], lam[y == 0], s=2, c="0.75", alpha=0.3, rasterized=True, label="y=0")
    ax.scatter(phi[y == 1], lam[y == 1], s=8, c="crimson", alpha=0.8, rasterized=True, label="y=1")
    ax.set_title(f"{g.safe_name}\ny+={int(y.sum())}/{len(y)}")
    ax.legend(markerscale=2, fontsize=8)

    # panel 1: masks
    ax = axes[1]
    ax.scatter(phi, lam, s=1, c="0.9", alpha=0.2, rasterized=True)
    ax.scatter(phi[tr & (y == 1)], lam[tr & (y == 1)], s=10, c="tab:blue", label="train∩y=1")
    ax.scatter(phi[tr & (y == 0)], lam[tr & (y == 0)], s=4, c="tab:cyan", alpha=0.4, label="train∩y=0")
    ax.scatter(phi[va], lam[va], s=12, c="tab:orange", label="val")
    ax.scatter(phi[te], lam[te], s=12, c="tab:green", label="test")
    ax.set_title(f"masks\ntr/va/te={tr.sum()}/{va.sum()}/{te.sum()}")
    ax.legend(markerscale=1.5, fontsize=8)

    # panel 2: edges by edge_y
    ax = axes[2]
    ax.scatter(phi[y == 0], lam[y == 0], s=1, c="0.85", alpha=0.2, rasterized=True)
    ax.scatter(phi[y == 1], lam[y == 1], s=6, c="crimson", alpha=0.7, rasterized=True)
    src, dst = g.edge_index.numpy()
    ey = g.edge_y.numpy()
    rng = np.random.default_rng(seed)
    idx = np.arange(len(ey))
    if len(idx) > max_edges:
        # keep all positive edges, subsample negatives
        pos = idx[ey == 1]
        neg = idx[ey == 0]
        take_neg = rng.choice(neg, size=min(len(neg), max_edges - len(pos)), replace=False) if len(neg) else neg
        idx = np.concatenate([pos, take_neg])
    for e in idx:
        i, j = src[e], dst[e]
        ax.plot(
            [phi[i], phi[j]],
            [lam[i], lam[j]],
            color=("darkred" if ey[e] == 1 else "steelblue"),
            alpha=(0.7 if ey[e] == 1 else 0.15),
            lw=(1.2 if ey[e] == 1 else 0.4),
        )
    ax.set_title(f"edges\nedge_y+= {int((ey==1).sum())}/{len(ey)}")

    for ax in axes:
        ax.set_xlabel("phi")
        ax.set_ylabel("lam")
        ax.set_aspect("equal", adjustable="datalim")
    fig.tight_layout()
    return g


# Prefer a MOCK with positives, then a named stream
prefer = catalog.sort_values(["edge_y_pos", "y_pos"], ascending=False)["file"].tolist()
sample_files = []
for name in prefer:
    if name not in sample_files:
        sample_files.append(name)
    if len(sample_files) >= 3:
        break

for name in sample_files:
    plot_graph_labels(DATA_DIR / name)
    plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'streams_per_region_iter3_pm_stepping2_k4_edges_pmcoh/stream_MOCK_2067.parquet'

## Edge-label deep dive on one suspicious graph

If `edge_y` is all zero, check whether that is because PM-coherence pruning removed all SS edges, or because labels never matched.

In [ ]:
suspects = catalog.loc[(catalog["y_pos"] > 0) & (catalog["edge_y_pos"] == 0), "file"].tolist()
focus_name = suspects[0] if suspects else catalog.sort_values("y_pos", ascending=False).iloc[0]["file"]
g = load_graph(DATA_DIR / focus_name)
print("focus:", focus_name, "stream_label=", g.stream_label)

y = g.y.numpy()
src, dst = g.edge_index.numpy()
print("nodes y=1:", int(y.sum()))
print("unique edge endpoint indices touching y=1:", len(np.unique(np.concatenate([src[y[src]==1], dst[y[dst]==1]]))))
print("degree of y=1 nodes (in this graph):")
deg = np.zeros(len(y), dtype=np.int64)
np.add.at(deg, src, 1)
np.add.at(deg, dst, 1)
print("  min/median/max among y=1:", deg[y == 1].min() if y.any() else None, np.median(deg[y == 1]) if y.any() else None, deg[y == 1].max() if y.any() else None)
print("  isolated y=1 nodes:", int(((y == 1) & (deg == 0)).sum()))

# Compare against unpruned / other variants if present
safe = g.safe_name
variants = sorted(PARENT.glob(f"stream_{safe}_E*_edge_index.pt"))
print(f"\nedge_index variants for {safe} ({len(variants)}):")
for p in variants:
    ei = torch.load(p, map_location="cpu", weights_only=True).long()
    s, t = ei[0].numpy(), ei[1].numpy()
    ss = int(((y[s] == 1) & (y[t] == 1)).sum())
    so = int((((y[s] == 1) & (y[t] == 0)) | ((y[s] == 0) & (y[t] == 1))).sum())
    bb = int(((y[s] == 0) & (y[t] == 0)).sum())
    print(f"  {p.name:70s} E={ei.size(1):6d}  SS/SO/BB={ss}/{so}/{bb}")

## Quick summary plot across the catalog

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

ax = axes[0]
ax.scatter(catalog["y_pos"], catalog["edge_y_pos"], c=np.log10(catalog["n_edges"].clip(lower=1)), cmap="viridis", s=40)
ax.set_xlabel("node positives (y=1)")
ax.set_ylabel("edge positives (edge_y=1)")
ax.set_title("positives: nodes vs edges")
ax.set_xscale("symlog")
ax.set_yscale("symlog")

ax = axes[1]
fracs = catalog[["edges_SS", "edges_SO", "edges_BB"]].div(catalog["n_edges"].replace(0, np.nan), axis=0)
ax.hist(fracs["edges_SS"].dropna(), bins=20, alpha=0.7, label="SS")
ax.hist(fracs["edges_SO"].dropna(), bins=20, alpha=0.7, label="SO")
ax.hist(fracs["edges_BB"].dropna(), bins=20, alpha=0.7, label="BB")
ax.set_xlabel("fraction of edges")
ax.set_title("edge endpoint class mix")
ax.legend()

ax = axes[2]
ax.scatter(catalog["y_pos_in_train"], catalog["y_pos_in_val"] + catalog["y_pos_in_test"], s=40)
ax.set_xlabel("y=1 in train")
ax.set_ylabel("y=1 in val+test")
ax.set_title("positive split")
ax.set_xscale("symlog")
ax.set_yscale("symlog")

fig.tight_layout()
plt.show()

print(
    catalog[["file", "y_pos", "edge_y_pos", "edges_SS", "edges_SO", "edges_BB", "y_pos_unmasked"]]
    .sort_values("edge_y_pos")
    .to_string(index=False)
)